# Adaptive Experimentation Loop with Ax

This tutorial demonstrates how to use **Ax** for Bayesian optimization using a plant growth example.

**Ax** is Facebook's Python library for managing adaptive experiments. It intelligently suggests which experiments to run next based on previous results, rather than randomly trying parameter combinations.

**The Process:**
- Ax proposes experiments (water/sunlight combinations)
- We "run these experiments" and measure plant growth
- We input the results back into Ax
- Ax learns from the data and proposes better experiments for the next round

In [ ]:
import os

import ax

%matplotlib inline

## Configure Optimization Run

These parameters control how the optimization will run:
- **NUM_ROUNDS**: Total number of optimization rounds (each round suggests multiple experiments)
- **NUM_PARALLEL_EXPERIMENTS**: How many experiments Ax suggests per round
- **BO_STATE_FILE**: Checkpoint file to save/resume optimization progress (prevents losing work if interrupted)

In [ ]:
NUM_ROUNDS = 8  # How many rounds of experiments to run
NUM_PARALLEL_EXPERIMENTS = 3  # How many experiments to run in parallel
BO_STATE_FILE = "bo_state.json"  # Where to save a checkpoint of the Bayesian optimization state

## Load the Ax Client

The **Client** is Ax's main interface that orchestrates the entire optimization process. It:
- Manages experiments and trials
- Decides which parameter combinations to try next
- Stores the history of all experiments and results
- Handles the machine learning models that power the optimization

**Loading from checkpoint**: If a checkpoint file exists (from previous runs), we load it to resume where we left off. Otherwise, we create a fresh client. This allows you to continue optimization across multiple sessions without losing progress.

In [ ]:
if os.path.exists(BO_STATE_FILE):
    client = ax.api.client.Client.load_from_json_file(BO_STATE_FILE)
client = ax.api.client.Client()

### Fill in Missing Experimental Values

**What are "running trials"?** When Ax suggests experiments, it marks them as "RUNNING" until you provide the results.

**Why this function is needed**: If your code crashes or you exit before completing all trials, Ax remembers which experiments were suggested but never completed. This function finds those incomplete trials and prompts you to enter the missing results.

**How it works**: The function checks the client's trial summary for any trials with status "RUNNING", then prompts you to input the results for each one. After entering results, it saves the updated state to the checkpoint file.

In [ ]:
def check_for_running_trials(client):
    """Check if there are any running trials and return their indices."""
    df = client.summarize()
    for _, trial in df[df["trial_status"] == "RUNNING"].iterrows():
        result_str = input(f"Enter result for trial {trial['trial_index']:02d}:\twater={trial['water']:.3f}\tsunlight={trial['sunlight']:.3f}")
        result = float(result_str)
        client.complete_trial(trial["trial_index"], {"plant_growth": result})
        client.save_to_json_file(BO_STATE_FILE)


check_for_running_trials(client)

## Define the Optimization Parameters

**Parameters** are the input variables you want to optimize. Each parameter needs:

- **name**: A unique identifier (e.g., "water", "sunlight")
- **parameter_type**: The data type - usually "float" for continuous values, but can be "int" or "str"
- **bounds**: The valid range for this parameter as a tuple (min_value, max_value)

In our example:
- **water**: Continuous values between 0.0 and 1.0 (representing frequency/amount)
- **sunlight**: Continuous values between 0.0 and 1.0 (representing exposure level)

These bounds constrain where Ax will search - it will never suggest values outside these ranges.

In [ ]:
parameters = [
    ax.api.configs.RangeParameterConfig(
        name="water", parameter_type="float", bounds=(0.0, 1.0),
    ),
    ax.api.configs.RangeParameterConfig(
        name="sunlight", parameter_type="float", bounds=(0.0, 1.0),
    ),
]
client.configure_experiment(parameters=parameters)

## Define the Objective Function

The **objective** is what you want to optimize (maximize or minimize). You need to:

1. **Define a metric name**: A string identifier for your measurement (e.g., "plant_growth")
2. **Specify the objective**: Tell Ax whether to maximize or minimize this metric

**Important notes:**
- By default, Ax **maximizes** the objective (so higher values = better)
- If you want to minimize something (like error rate), you can negate it or use `minimize` in the objective
- The metric name must match exactly what you'll use when reporting results with `complete_trial()`

In our example, we want to maximize plant growth, so higher plant_growth values are better.

In [ ]:
metric_name = "plant_growth"
objective = f"{metric_name}"  # A simple expression for the objective
client.configure_optimization(objective=objective)

## (Advanced) Configure the Generation Strategy

The **generation strategy** controls how Ax chooses parameter combinations to try. By default, Ax uses a good strategy, but you can customize it:

**Two-phase approach:**
1. **Sobol (quasi-random)**: First few rounds use structured random sampling to explore the parameter space
2. **Bayesian Optimization**: Subsequent rounds use machine learning to intelligently pick promising parameter combinations

**Why start with Sobol?**
- Provides good initial coverage of the parameter space
- Gives the Bayesian model good training data to learn from
- More reliable than pure random sampling

**When to customize:**
- You can skip this section for simple use cases (Ax's default is usually fine)
- Customize when you need specific exploration vs exploitation behavior
- The threshold (`NUM_PARALLEL_EXPERIMENTS`) determines when to switch from Sobol to BO

In [ ]:
bo_node = ax.generation_strategy.generation_node.GenerationNode(
    node_name="BO",
    model_specs=[
        ax.generation_strategy.model_spec.GeneratorSpec(
            model_enum=ax.modelbridge.registry.Generators.BOTORCH_MODULAR,
        )
    ]
)

sobol_node = ax.generation_strategy.generation_node.GenerationNode(
    node_name="Sobol",
    model_specs=[
        ax.generation_strategy.model_spec.GeneratorSpec(
            model_enum=ax.modelbridge.registry.Generators.SOBOL,
        )
    ],
    transition_criteria=[
        ax.generation_strategy.transition_criterion.MinTrials(
            threshold=(NUM_PARALLEL_EXPERIMENTS),
            transition_to=bo_node.node_name,
            use_all_trials_in_exp=True,
        )
    ]
)

client.set_generation_strategy(
    generation_strategy=ax.generation_strategy.generation_strategy.GenerationStrategy(
        name="2xSobol + BO",
        nodes=[sobol_node, bo_node],
    )
)

## Run the Optimization Loop

This is the main optimization workflow that repeats for each round:

**For each round:**
1. **Get suggestions**: `client.get_next_trials()` asks Ax to suggest the next batch of experiments
2. **Display experiments**: Shows the parameter combinations Ax wants you to try
3. **Save checkpoint**: Saves current state before running experiments (in case of crashes)
4. **Run experiments**: For each suggested trial, you input the measured result
5. **Complete trials**: `client.complete_trial()` records the results in Ax
6. **Update checkpoint**: Saves the new results

**Key concepts:**
- **Trial**: A single experiment with specific parameter values
- **Round**: A batch of trials that run in parallel
- **Trial index**: Unique identifier for each experiment (useful for tracking)

**Interactive workflow**: The code prompts you to input results manually, simulating real experiments where you'd measure outcomes in a lab or production system.

In [ ]:
for round in range(NUM_ROUNDS):
    trials = client.get_next_trials(max_trials=NUM_PARALLEL_EXPERIMENTS)

    print(f"Round {round + 1} of {NUM_ROUNDS}:\nExperiments:")
    for trial_index, parameters in trials.items():
        water = parameters["water"]
        sunlight = parameters["sunlight"]
        print(f" - Trial {trial_index:02d}:\twater={water:.3f}\tsunlight={sunlight:.3f}")

    client.save_to_json_file(BO_STATE_FILE)

    for trial_index, parameters in trials.items():
        # Simulate a trial with random data
        water = parameters["water"]
        sunlight = parameters["sunlight"]
        plant_growth_str = input(f"Input plant height for Trial {trial_index} (water={water:.3f}, sunlight={sunlight:.3f}): ")
        print(plant_growth_str)
        plant_growth = float(plant_growth_str)

        # Record the result
        client.complete_trial(trial_index, {metric_name: plant_growth})
        print(f"  * Completed trial {trial_index:02d} with:\twater={water:.3f}\tsunlight={sunlight:.3f}\tplant_growth={plant_growth:.3f}")
        client.save_to_json_file(BO_STATE_FILE)

    print()

## View Results

Now you can analyze the optimization results:

**Summary table**: `client.summarize()` shows all trials with:
- **trial_index**: Unique identifier for each experiment
- **trial_status**: Whether completed, running, or failed
- **generation_node**: Which strategy generated this trial (Sobol or BO)
- **plant_growth**: The measured outcome
- **water/sunlight**: The parameter values that were tested

**Analysis cards**: `client.compute_analyses()` generates interactive visualizations showing:
- Parameter importance (which inputs matter most)
- Response surface plots (how the objective changes across parameter space)
- Optimization progress over time
- Best parameter combinations found

**What to look for:**
- Trials with highest plant_growth values show the best parameter combinations
- Notice how later trials (BO phase) tend to cluster around promising regions
- The "generation_node" column shows the transition from Sobol exploration to BO exploitation

In [ ]:
client.summarize()

In [ ]:
cards = client.compute_analyses(display=True)